In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [14]:
import os
base = '/content/drive/MyDrive/ISRO/insat/'
total = 0
for product in ['LST', 'IMC', 'SST']:
    path = base + product + '/'
    count = len(os.listdir(path)) if os.path.exists(path) else 0
    total += count
    print(f"{product}: {count} files")
print(f"Total: {total} files")

LST: 1520 files
IMC: 1375 files
SST: 1388 files
Total: 4283 files


In [4]:
import zipfile
import os

# Extract project
with zipfile.ZipFile(
    '/content/drive/MyDrive/digital_twin.zip', 'r'
) as z:
    z.extractall('/content/')

print("✅ Project extracted!")
os.listdir('/content/digital_twin/')

✅ Project extracted!


['run.py',
 'graph',
 'model',
 'requirements.txt',
 'data',
 'evaluation',
 'config.yaml',
 'visualization',
 'training',
 'download_mosdac.sh',
 'graphcast_india.log',
 'setup.sh']

In [6]:
!pip install torch torchvision torchaudio \
  --index-url https://download.pytorch.org/whl/cu121 -q
print("✅ PyTorch done!")

✅ PyTorch done!


In [7]:
!pip install torch-geometric -q
!pip install torch-scatter torch-sparse \
  -f https://data.pyg.org/whl/torch-2.5.1+cu121.html -q
print("✅ PyG done!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 116.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 3.7 MB/s eta 0:00:00
✅ PyG done!


In [8]:
!pip install numpy pandas xarray h5py netCDF4 scipy \
             tqdm pyyaml rich scikit-learn \
             matplotlib seaborn -q
print("✅ Core packages done!")

✅ Core packages done!


In [9]:
!pip install gradio -q
!pip install cartopy -q
print("✅ UI packages done!")

✅ UI packages done!


In [10]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM:", torch.cuda.get_device_properties(0).total_memory/1e9, "GB")

CUDA available: True
GPU: Tesla T4
VRAM: 15.637086208 GB


In [11]:
import os

# Create symlinks to data already in Drive
# No need to copy — use directly from Drive!
os.makedirs('/content/digital_twin/data/raw', exist_ok=True)

# Link IMD data
!ln -sf '/content/drive/MyDrive/ISRO/imd' \
  '/content/digital_twin/data/raw/imd'

# Link INSAT data
!ln -sf '/content/drive/MyDrive/ISRO/insat' \
  '/content/digital_twin/data/raw/insat'

print("✅ Data linked!")
!ls /content/digital_twin/data/raw/

✅ Data linked!
imd  insat


In [16]:
import os

insat_base = '/content/digital_twin/data/raw/insat/'
os.makedirs(insat_base, exist_ok=True)

for product in ['LST', 'IMC', 'SST']:
    dst = insat_base + product          # where code looks
    src = f'/content/drive/MyDrive/ISRO/insat/{product}'  # where files are

    # Remove broken symlink if exists
    if os.path.islink(dst):
        os.unlink(dst)

    if os.path.exists(src):
        os.symlink(src, dst)            # src → dst (fixed order)
        days = [d for d in os.listdir(dst) if os.path.isdir(dst+'/'+d)]
        print(f"✅ {product}: symlink created! ({len(days)} days)")
    else:
        print(f"❌ {product}: not found in Drive at {src}")
        # Show what IS in the ISRO folder
        isro_path = '/content/drive/MyDrive/ISRO/'
        if os.path.exists(isro_path):
            print(f"   Drive ISRO contents: {os.listdir(isro_path)}")

✅ LST: symlink created! (0 days)
✅ IMC: symlink created! (0 days)
✅ SST: symlink created! (0 days)


In [17]:
import os

base = '/content/drive/MyDrive/ISRO/insat/'
for product in ['LST', 'IMC', 'SST']:
    path = base + product + '/'
    items = os.listdir(path)[:5]  # first 5 items
    print(f"\n{product} sample contents:")
    for item in items:
        full = path + item
        print(f"  {'DIR' if os.path.isdir(full) else 'FILE'}: {item}")


LST sample contents:
  FILE: 3RIMG_11MAY2022_1115_L2B_LST_V01R00.h5
  FILE: 3RIMG_11MAY2022_1107_L2B_LST_V01R00.h5
  FILE: 3RIMG_11MAY2022_1119_L2B_LST_V01R00.h5
  FILE: 3RIMG_11MAY2022_1124_L2B_LST_V01R00.h5
  FILE: 3RIMG_11MAY2022_1102_L2B_LST_V01R00.h5

IMC sample contents:
  FILE: 3RIMG_09FEB2023_1115_L2B_IMC_V01R00.h5
  FILE: 3RIMG_09FEB2022_1115_L2B_IMC_V01R00.h5
  FILE: 3RIMG_09FEB2022_1145_L2B_IMC_V01R00.h5
  FILE: 3RIMG_09DEC2023_1145_L2B_IMC_V01R00.h5
  FILE: 3RIMG_09JAN2022_1115_L2B_IMC_V01R00.h5

SST sample contents:
  FILE: 3RIMG_09JAN2023_1145_L2B_SST_V02R00.h5
  FILE: 3RIMG_09JUL2022_1145_L2B_SST_V02R00.h5
  FILE: 3RIMG_09JUN2022_1115_L2B_SST_V02R00.h5
  FILE: 3RIMG_09JUN2022_1145_L2B_SST_V02R00.h5
  FILE: 3RIMG_09JUL2023_1145_L2B_SST_V02R00.h5


In [18]:
import os
import shutil
import re

MONTHS = {
    "JAN":"01","FEB":"02","MAR":"03","APR":"04",
    "MAY":"05","JUN":"06","JUL":"07","AUG":"08",
    "SEP":"09","OCT":"10","NOV":"11","DEC":"12"
}

base = '/content/drive/MyDrive/ISRO/insat/'
moved = 0
skipped = 0

for product in ['LST', 'IMC', 'SST']:
    src_dir = base + product + '/'
    files = os.listdir(src_dir)
    print(f"\nOrganizing {product}: {len(files)} files...")

    for filename in files:
        # Parse: 3RIMG_01JAN2023_1115_L2B_LST_V01R00.h5
        m = re.match(
            r"3RIMG_(\d{2})([A-Z]{3})(\d{4})_\d{4}_L2B_\w+_V\d+R\d+\.h5",
            filename
        )
        if not m:
            skipped += 1
            continue

        day, mon, year = m.group(1), m.group(2), m.group(3)

        # Only keep 2023
        if year != "2023":
            skipped += 1
            continue

        date_str = f"{year}{MONTHS[mon]}{day}"
        out_dir  = f"{src_dir}{date_str}/"
        os.makedirs(out_dir, exist_ok=True)

        src = src_dir + filename
        dst = out_dir + filename

        if not os.path.exists(dst):
            shutil.move(src, dst)
            moved += 1

print(f"\n✅ Done!")
print(f"   Moved : {moved} files into date folders")
print(f"   Skipped: {skipped} files (2022 or unrecognized)")
print(f"\nFinal structure:")
for product in ['LST', 'IMC', 'SST']:
    days = [d for d in os.listdir(base+product+'/')
            if os.path.isdir(base+product+'/'+d)]
    print(f"  {product}: {len(days)} days")


Organizing LST: 1520 files...

Organizing IMC: 1375 files...

Organizing SST: 1388 files...

✅ Done!
   Moved : 2051 files into date folders
   Skipped: 2232 files (2022 or unrecognized)

Final structure:
  LST: 343 days
  IMC: 342 days
  SST: 343 days


In [19]:
import os

insat_base = '/content/digital_twin/data/raw/insat/'
os.makedirs(insat_base, exist_ok=True)

for product in ['LST', 'IMC', 'SST']:
    dst = insat_base + product
    src = f'/content/drive/MyDrive/ISRO/insat/{product}'

    # Remove broken symlink if exists
    if os.path.islink(dst):
        os.unlink(dst)
    if os.path.exists(dst):
        shutil.rmtree(dst)

    os.symlink(src, dst)
    days = [d for d in os.listdir(dst) if os.path.isdir(dst+'/'+d)]
    print(f"✅ {product}: linked! ({len(days)} days)")

✅ LST: linked! (343 days)
✅ IMC: linked! (342 days)
✅ SST: linked! (343 days)


In [22]:
insat_reader = open('/content/digital_twin/data/insat_reader.py').read()

# Fix 1: Update product keys to match actual HDF5 structure
old_keys = '''    product_keys = {
        "LST": "L2B_LST",
        "SST": "L2B_SST",
        "IMC": "L2B_IMC",
    }'''

new_keys = '''    product_keys = {
        "LST": "LST",
        "SST": "SST",
        "IMC": "IMC",
    }'''

insat_reader = insat_reader.replace(old_keys, new_keys)

# Fix 2: Handle 3D data (time, lat, lon) → squeeze to 2D
old_data = '''            data = f[key][:]'''

new_data = '''            data = f[key][:]
            # Squeeze time dimension if 3D
            if data.ndim == 3:
                data = data[0]  # take first time slice'''

insat_reader = insat_reader.replace(old_data, new_data)

open('/content/digital_twin/data/insat_reader.py', 'w').write(insat_reader)
print("✅ insat_reader.py fixed!")

✅ insat_reader.py fixed!


In [23]:
# Quick test on one file
import h5py, os

sample = '/content/drive/MyDrive/ISRO/insat/LST/20230101/'
files = os.listdir(sample)
print(f"Files in 20230101: {files}")

with h5py.File(sample + files[0], 'r') as f:
    print(f"\nKeys in HDF5: {list(f.keys())}")
    print(f"LST shape: {f['LST'][:].shape}")
    if 'Latitude' in f:
        print(f"Lat shape: {f['Latitude'][:].shape}")
    if 'GeoX' in f:
        print(f"GeoX shape: {f['GeoX'][:].shape}")

Files in 20230101: ['3RIMG_01JAN2023_1145_L2B_LST_V01R00.h5', '3RIMG_01JAN2023_1115_L2B_LST_V01R00.h5']

Keys in HDF5: ['GeoX', 'GeoY', 'LST', 'Latitude', 'Longitude', 'time']
LST shape: (1, 2816, 2805)
Lat shape: (2816, 2805)
GeoX shape: (2805,)


In [25]:
# STEP 1: Fix product keys
reader = open('/content/digital_twin/data/insat_reader.py').read()

old = '''    product_keys = {
        "LST": "L2B_LST",
        "SST": "L2B_SST",
        "IMC": "L2B_IMC",
    }'''

new = '''    product_keys = {
        "LST": "LST",
        "SST": "SST",
        "IMC": "IMC",
    }'''

reader = reader.replace(old, new)

# STEP 2: Fix 3D to 2D squeeze
old2 = '''            data = f[key][:]

            # Get scale, offset, fill from attributes'''

new2 = '''            data = f[key][:]
            if data.ndim == 3:
                data = data[0]  # squeeze (1,lat,lon) → (lat,lon)

            # Get scale, offset, fill from attributes'''

reader = reader.replace(old2, new2)

# STEP 3: Fix 2D lat/lon extraction
old3 = '''            if "Latitude" in f and "Longitude" in f:
                lats = f["Latitude"][:]
                lons = f["Longitude"][:]
                # INSAT lat/lon can be 2D (y, x) — take first row/col
                if lats.ndim == 2:
                    lats = lats[:, 0]
                    lons = lons[0, :]'''

new3 = '''            if "Latitude" in f and "Longitude" in f:
                lats = f["Latitude"][:]
                lons = f["Longitude"][:]
                if lats.ndim == 2:
                    lats = lats[:, 0]   # 2D → 1D column
                    lons = lons[0, :]   # 2D → 1D row
            elif "GeoX" in f and "GeoY" in f:
                lons = f["GeoX"][:]
                lats = f["GeoY"][:]'''

reader = reader.replace(old3, new3)

open('/content/digital_twin/data/insat_reader.py', 'w').write(reader)
print("✅ All 3 fixes applied!")

# Confirm
if '"LST": "LST"' in reader:
    print("✅ Fix 1: Product keys correct")
if 'data = data[0]' in reader:
    print("✅ Fix 2: 3D squeeze correct")
if 'lats = lats[:, 0]' in reader:
    print("✅ Fix 3: 2D lat/lon correct")

✅ All 3 fixes applied!
✅ Fix 1: Product keys correct
✅ Fix 2: 3D squeeze correct
✅ Fix 3: 2D lat/lon correct


In [26]:
import sys
sys.path.insert(0, '/content/digital_twin')

# Force reload module
import importlib
import data.insat_reader
importlib.reload(data.insat_reader)
from data.insat_reader import read_insat_file

sample_dir = '/content/drive/MyDrive/ISRO/insat/LST/20230101/'
import os
f = os.listdir(sample_dir)[0]
print(f"Testing file: {f}")

da = read_insat_file(sample_dir + f, 'LST')
if da is not None:
    print(f"✅ Read successfully!")
    print(f"   Shape: {da.shape}")
    print(f"   Lat range: {float(da.lat.min()):.1f} to {float(da.lat.max()):.1f}")
    print(f"   Lon range: {float(da.lon.min()):.1f} to {float(da.lon.max()):.1f}")
    print(f"   Value range: {float(da.min()):.1f} to {float(da.max()):.1f} °C")
else:
    print("❌ Still failing!")

Testing file: 3RIMG_01JAN2023_1145_L2B_LST_V01R00.h5
✅ Read successfully!
   Shape: (2816, 2805)
   Lat range: 32767.0 to 32767.0
   Lon range: 32767.0 to 32767.0
   Value range: 238.6 to 323.0 °C


In [27]:
reader = open('/content/digital_twin/data/insat_reader.py').read()

# Fix lat/lon — use GeoX/GeoY instead of Latitude/Longitude
# (Latitude/Longitude arrays contain fill values in this format)
old = '''            if "Latitude" in f and "Longitude" in f:
                lats = f["Latitude"][:]
                lons = f["Longitude"][:]
                if lats.ndim == 2:
                    lats = lats[:, 0]   # 2D → 1D column
                    lons = lons[0, :]   # 2D → 1D row
            elif "GeoX" in f and "GeoY" in f:
                lons = f["GeoX"][:]
                lats = f["GeoY"][:]'''

new = '''            # Use GeoX/GeoY — Latitude/Longitude contain fill values
            if "GeoX" in f and "GeoY" in f:
                lons = f["GeoX"][:]   # 1D longitude array
                lats = f["GeoY"][:]   # 1D latitude array
            elif "Latitude" in f and "Longitude" in f:
                lats = f["Latitude"][:]
                lons = f["Longitude"][:]
                if lats.ndim == 2:
                    lats = lats[:, 0]
                    lons = lons[0, :]'''

reader = reader.replace(old, new)

# Fix Kelvin to Celsius conversion
old2 = '''            data = data * scale + offset'''

new2 = '''            data = data * scale + offset
            # Convert Kelvin to Celsius if values look like Kelvin
            if float(np.nanmean(data)) > 200:
                data = data - 273.15'''

reader = reader.replace(old2, new2)

open('/content/digital_twin/data/insat_reader.py', 'w').write(reader)
print("✅ Lat/Lon and Kelvin fixes applied!")

✅ Lat/Lon and Kelvin fixes applied!


In [28]:
importlib.reload(data.insat_reader)
from data.insat_reader import read_insat_file

da = read_insat_file(sample_dir + f, 'LST')
if da is not None:
    print(f"✅ Shape: {da.shape}")
    print(f"   Lat range: {float(da.lat.min()):.1f} to {float(da.lat.max()):.1f}")
    print(f"   Lon range: {float(da.lon.min()):.1f} to {float(da.lon.max()):.1f}")
    print(f"   Temp range: {float(da.min()):.1f} to {float(da.max()):.1f} °C")

✅ Shape: (2816, 2805)
   Lat range: 0.0 to 2815.0
   Lon range: 0.0 to 2804.0
   Temp range: -34.5 to 49.9 °C


Temperature is Fixed! ✅ (-34.5 to 49.9°C — perfect!)
But GeoX/GeoY are pixel indices (0-2815), not real coordinates. We need to compute real lat/lon from the HDF5 metadata.

In [29]:
import h5py
import numpy as np

sample_dir = '/content/drive/MyDrive/ISRO/insat/LST/20230101/'
import os
f = os.listdir(sample_dir)[0]

with h5py.File(sample_dir + f, 'r') as hf:
    # Print ALL attributes of the file
    print("=== File attributes ===")
    for k, v in hf.attrs.items():
        print(f"  {k}: {v}")

    print("\n=== GeoX attributes ===")
    for k, v in hf['GeoX'].attrs.items():
        print(f"  {k}: {v}")

    print("\n=== GeoY attributes ===")
    for k, v in hf['GeoY'].attrs.items():
        print(f"  {k}: {v}")

    print("\n=== LST attributes ===")
    for k, v in hf['LST'].attrs.items():
        print(f"  {k}: {v}")

    print("\n=== Latitude sample values ===")
    lat = hf['Latitude'][:]
    print(f"  Shape: {lat.shape}")
    print(f"  Unique values count: {len(np.unique(lat))}")
    print(f"  Sample corner values:")
    print(f"    [0,0]={lat[0,0]:.2f}  [0,-1]={lat[0,-1]:.2f}")
    print(f"    [-1,0]={lat[-1,0]:.2f}  [-1,-1]={lat[-1,-1]:.2f}")

=== File attributes ===
  Acquisition_Date: b'01JAN2023'
  Acquisition_End_Time: b'01-JAN-2023T12:12:20'
  Acquisition_Start_Time: b'01-JAN-2023T11:45:26'
  Acquisition_Time_in_GMT: b'1145'
  Ground_Station: b'BES,SAC/ISRO,Ahmedabad,INDIA.'
  HDF_Product_File_Name: b'3RIMG_01JAN2023_1145_L2B_LST_V01R00.h5'
  Nominal_Altitude(km): [36000.]
  Nominal_Central_Point_Coordinates(degrees)_Latitude_Longitude: [ 0. 74.]
  Output_Format: b'hdf5-1.8.8'
  Processing_Level: b'L2B'
  Product_Creation_Time: b'2023-01-01T17:48:28'
  Product_Type: b'GEOPHY'
  Satellite_Name: b'INSAT-3DR'
  Sensor_Id: b'IMG'
  Sensor_Name: b'IMAGER'
  Software_Version: b'1.0'
  Station_Id: b'BES'
  Unique_Id: b'3RIMG_01JAN2023_1145'
  conventions: b'CF-1.6'
  institute: b'BES,SAC/ISRO,Ahmedabad,INDIA.'
  left_longitude: [-7.1567035]
  lower_latitude: [-81.04153]
  right_longitude: [155.15671]
  source: b'BES,SAC/ISRO,Ahmedabad,INDIA.'
  title: b'3RIMG_01JAN2023_1145_L2B'
  upper_latitude: [81.04153]

=== GeoX attribute

In [30]:
reader = open('/content/digital_twin/data/insat_reader.py').read()

old = '''            # Use GeoX/GeoY — Latitude/Longitude contain fill values
            if "GeoX" in f and "GeoY" in f:
                lons = f["GeoX"][:]   # 1D longitude array
                lats = f["GeoY"][:]   # 1D latitude array
            elif "Latitude" in f and "Longitude" in f:
                lats = f["Latitude"][:]
                lons = f["Longitude"][:]
                if lats.ndim == 2:
                    lats = lats[:, 0]
                    lons = lons[0, :]'''

new = '''            # Compute real lat/lon from bounding box attributes
            attrs_file = dict(f.attrs)
            upper_lat  = float(attrs_file.get("upper_latitude",  [81.0])[0])
            lower_lat  = float(attrs_file.get("lower_latitude", [-81.0])[0])
            left_lon   = float(attrs_file.get("left_longitude",  [-7.16])[0])
            right_lon  = float(attrs_file.get("right_longitude", [155.16])[0])
            n_lat, n_lon = data.shape
            lats = np.linspace(upper_lat, lower_lat, n_lat)
            lons = np.linspace(left_lon,  right_lon, n_lon)'''

reader = reader.replace(old, new)
open('/content/digital_twin/data/insat_reader.py', 'w').write(reader)
print("✅ Real lat/lon fix applied!")

✅ Real lat/lon fix applied!


In [37]:
import os
import importlib
import data.insat_reader # Import the module initially to get its reference

# --- Start of file fixing logic ---
file_path = '/content/digital_twin/data/insat_reader.py'
current_file_content = open(file_path).read()

lines = current_file_content.splitlines()
cleaned_lines = []
i = 0
while i < len(lines):
    # This specific 'else:' is the problematic one due to its previous replacement.
    # It has 12 spaces indentation.
    if lines[i] == '            else:' and i + 3 < len(lines):
        # Check if the subsequent lines match the problematic fallback logic
        # Indentations should be 16 spaces for these lines
        if (lines[i+1] == '                # Fallback: INSAT-3DR covers roughly 60°E–105°E, -10°N–45°N' and
            lines[i+2].startswith('                lats = np.linspace(-10, 45,') and
            lines[i+3].startswith('                lons = np.linspace( 60, 105,')):
            # This is the problematic block, skip these 4 lines
            i += 4
            continue # Skip adding these lines to cleaned_lines
    cleaned_lines.append(lines[i])
    i += 1

fixed_content = "\n".join(cleaned_lines)

# Write the cleaned content back to the file
with open(file_path, 'w') as f_out:
    f_out.write(fixed_content)
# --- End of file fixing logic ---

# Now that the file is syntactically correct, proceed with loading and testing
# Reload the module to load the fixed content
importlib.reload(data.insat_reader)
from data.insat_reader import read_insat_file

# sample_dir and f are defined in previous cells, assuming they are still in kernel state
# from previous execution of 5l6NiSp_UWdH
da = read_insat_file(sample_dir + f, 'LST')
if da is not None:
    print(f"✅ Shape: {da.shape}")
    print(f"   Lat range: {float(da.lat.min()):.1f} to {float(da.lat.max()):.1f}")
    print(f"   Lon range: {float(da.lon.min()):.1f} to {float(da.lon.max()):.1f}")
    print(f"   Temp range: {float(da.min()):.1f} to {float(da.max()):.1f} °C")
else:
    print("❌ Failed to read file after fix.")

✅ Shape: (2816, 2805)
   Lat range: -81.0 to 81.0
   Lon range: -7.2 to 155.2
   Temp range: -34.5 to 49.9 °C


In [41]:
import os
os.chdir('/content/digital_twin')

# Read the current content of insat_reader.py
file_path = '/content/digital_twin/data/insat_reader.py'
current_insat_reader_content = open(file_path).read()

# Define the old code block that needs modification
# This includes the `continue` from the previous fix
old_code_block = '''        da = da.sel(
            lat=slice(target_lats.min() - 1, target_lats.max() + 1),
            lon=slice(target_lons.min() - 1, target_lons.max() + 1),
        )
        # Check if the clipped data array is empty
        if da.size == 0:
            import logging
            log = logging.getLogger(__name__)
            log.debug(f"Clipped INSAT data for {f} is empty. Skipping interpolation.")
            continue
        # Regrid to IMD grid
        da_regrid = da.interp(
            lat=target_lats, lon=target_lons, method="linear"
        )
        slices.append(da_regrid.values)'''

# Define the new code block with added logging
new_code_block = '''        da = read_insat_file(f, product)
        if da is None:
            log.debug(f"read_insat_file returned None for {f}")
            continue
        # Clip to India domain
        initial_shape = da.shape
        da = da.sel(
            lat=slice(target_lats.min() - 1, target_lats.max() + 1),
            lon=slice(target_lons.min() - 1, target_lons.max() + 1),
        )
        if da.size == 0:
            log.debug(f"Clipped INSAT data from {initial_shape} to {da.shape} (empty) for {f}. Skipping interpolation.")
            continue
        log.debug(f"Clipped INSAT data from {initial_shape} to {da.shape} for {f}")
        # Regrid to IMD grid
        da_regrid = da.interp(
            lat=target_lats, lon=target_lons, method="linear"
        )
        slices.append(da_regrid.values)'''

# Replace the old block with the new block
modified_insat_reader_content = current_insat_reader_content.replace(old_code_block, new_code_block)

# Additionally, update the `product_keys` again to ensure it's correct
# This is a safe guard in case previous replace operations were not fully clean.
prod_keys_old = '''    product_keys = {
        "LST": "L2B_LST",
        "SST": "L2B_SST",
        "IMC": "L2B_IMC",
    }'''
prod_keys_new = '''    product_keys = {
        "LST": "LST",
        "SST": "SST",
        "IMC": "IMC",
    }'''
modified_insat_reader_content = modified_insat_reader_content.replace(prod_keys_old, prod_keys_new)

# Write the modified content back to the file
with open(file_path, 'w') as f_out:
    f_out.write(modified_insat_reader_content)

print("✅ insat_reader.py updated with detailed logging!")

# Now run the python script
!python run.py --mode train --real_data --year 2023

✅ insat_reader.py updated with detailed logging!
╭─────────────────────────────────────────╮
│ 🌧️  Mini-GraphCast India                 │
│ PS-5 Bharatiya Antariksh Hackathon 2026 │
│ Mode: train | Data: real                │
╰─────────────────────────────────────────╯
Device: cuda
GPU: Tesla T4
VRAM: 15.6 GB

Step 1: Loading data...
09:41:55 | INFO | run | Loading real IMD data for year 2023...
09:41:55 | INFO | numexpr.utils | NumExpr defaulting to 2 threads.
09:41:56 | INFO | data.imd_reader | Loaded IMD rainfall: (365, 129, 135) for year 2023
09:41:56 | INFO | data.imd_reader | Loaded IMD max temp: (365, 31, 31) for year 2023
09:41:56 | INFO | data.imd_reader | Loaded IMD min temp: (365, 31, 31) for year 2023
09:41:57 | INFO | run | Loading real INSAT data for year 2023...
09:41:57 | WARNING | data.insat_reader | No valid INSAT LST files for this day — filling NaN
09:42:04 | WARNING | data.insat_reader | No valid INSAT LST files for this day — filling NaN
^C


In [1]:
import torch
import torch.nn as nn

X = torch.randn(32, 10)
layer = nn.Linear(10, 5)

out = layer(X)
print(out.shape)

torch.Size([32, 5])


In [3]:
import torch

pred = torch.tensor([1,2,3])
true = torch.tensor([1,3,4])

# Convert the boolean tensor to float before calculating the mean
print((pred==true).float().mean().item())

0.3333333432674408


In [1]:
import h5py
import numpy as np
import pandas as pd
import datetime
import os
import re
from pathlib import Path
from tqdm import tqdm
from scipy.interpolate import RegularGridInterpolator
import argparse
import sys

# ══════════════════════════════════════════════════════════════
# CONFIGURATION — Update these paths for your environment
# ══════════════════════════════════════════════════════════════

YEAR        = 2023
IMD_DIR     = '/content/drive/MyDrive/ISRO/imd'
INSAT_DIR   = '/content/drive/MyDrive/ISRO/insat'
OUTPUT_CSV  = f'/content/drive/MyDrive/ISRO/master_dataset_{YEAR}.csv'

# ── IMD Grid Definitions (VERIFIED from actual files) ─────────
LATS_RAIN = np.round(np.arange(6.5,  38.75, 0.25), 2)   # 129 values
LONS_RAIN = np.round(np.arange(66.5, 100.25, 0.25), 2)  # 135 values
LATS_TEMP = np.round(np.arange(7.5,  38.5,  1.0),  2)   # 31 values
LONS_TEMP = np.round(np.arange(67.5, 98.5,  1.0),  2)   # 31 values

# ── India bounding box for INSAT clipping ────────────────────
LAT_MIN, LAT_MAX = 6.5,  38.5
LON_MIN, LON_MAX = 66.5, 100.0

# ══════════════════════════════════════════════════════════════
# PART 1 — READ IMD DATA
# ══════════════════════════════════════════════════════════════

def read_imd_data(year: int, imd_dir: str) -> tuple:
    """
    Read IMD rainfall + tmax + tmin binary files.

    Returns:
        rain    : (365, 129, 135) float64 — mm/day, NaN for ocean
        tmax_025: (365, 129, 135) float64 — °C, regridded to 0.25°
        tmin_025: (365, 129, 135) float64 — °C, regridded to 0.25°
    """
    n_days = 366 if _is_leap(year) else 365
    imd_dir = Path(imd_dir)

    # ── Rainfall (0.25° grid, fill = -999) ───────────────────
    rain_file = imd_dir / 'rain' / f'{year}.grd'
    print(f"  Reading rainfall: {rain_file}")
    raw  = np.fromfile(str(rain_file), dtype=np.float32)
    rain = raw.reshape(n_days, 129, 135).astype(np.float64)
    rain[rain == -999.0] = np.nan   # ocean / missing
    print(f"  Rainfall: {rain.shape}, range {np.nanmin(rain):.1f}"
          f"–{np.nanmax(rain):.1f} mm, valid pts/day: "
          f"{(~np.isnan(rain[0])).sum()}")

    # ── Max Temperature (1.0° grid, fill = 99.9) ─────────────
    tmax_file = imd_dir / 'tmax' / f'{year}.GRD'
    print(f"  Reading tmax: {tmax_file}")
    raw  = np.fromfile(str(tmax_file), dtype=np.float32)
    tmax = raw.reshape(n_days, 31, 31).astype(np.float64)
    tmax[tmax >= 99.9]  = np.nan
    tmax[tmax <= -99.9] = np.nan
    print(f"  Tmax: {tmax.shape}, range "
          f"{np.nanmin(tmax):.1f}–{np.nanmax(tmax):.1f} °C")

    # ── Min Temperature (1.0° grid, fill = 99.9) ─────────────
    tmin_file = imd_dir / 'tmin' / f'{year}.GRD'
    print(f"  Reading tmin: {tmin_file}")
    raw  = np.fromfile(str(tmin_file), dtype=np.float32)
    tmin = raw.reshape(n_days, 31, 31).astype(np.float64)
    tmin[tmin >= 99.9]  = np.nan
    tmin[tmin <= -99.9] = np.nan
    print(f"  Tmin: {tmin.shape}, range "
          f"{np.nanmin(tmin):.1f}–{np.nanmax(tmin):.1f} °C")

    # ── Regrid temperature to 0.25° grid ─────────────────────
    print("  Regridding temperature to 0.25° grid...")
    tmax_025 = np.full((n_days, 129, 135), np.nan)
    tmin_025 = np.full((n_days, 129, 135), np.nan)

    # Build query points once
    pts = np.array([[la, lo]
                    for la in LATS_RAIN
                    for lo in LONS_RAIN])

    for d in tqdm(range(n_days), desc="  Regridding"):
        # Tmax
        if not np.all(np.isnan(tmax[d])):
            interp = RegularGridInterpolator(
                (LATS_TEMP, LONS_TEMP), tmax[d],
                method='linear', bounds_error=False, fill_value=np.nan
            )
            tmax_025[d] = interp(pts).reshape(129, 135)

        # Tmin
        if not np.all(np.isnan(tmin[d])):
            interp = RegularGridInterpolator(
                (LATS_TEMP, LONS_TEMP), tmin[d],
                method='linear', bounds_error=False, fill_value=np.nan
            )
            tmin_025[d] = interp(pts).reshape(129, 135)

    print(f"  Tmax 0.25°: range "
          f"{np.nanmin(tmax_025):.1f}–{np.nanmax(tmax_025):.1f} °C")
    print(f"  Tmin 0.25°: range "
          f"{np.nanmin(tmin_025):.1f}–{np.nanmax(tmin_025):.1f} °C")

    return rain, tmax_025, tmin_025


# ══════════════════════════════════════════════════════════════
# PART 2 — READ INSAT DATA
# ══════════════════════════════════════════════════════════════

def read_one_insat_file(filepath: str, product: str) -> dict | None:
    """
    Read ONE INSAT HDF5 file → dict of India grid values.

    Returns dict: {(lat_snap, lon_snap): value_celsius}
    or None if file unreadable.

    VERIFIED structure:
      - Data key: 'LST', 'SST', 'IMC' (NOT 'L2B_LST' etc)
      - Shape: (1, 2816, 2805) — squeeze time dim
      - Lat/Lon 2D arrays × 0.01 scale factor
      - Units: Kelvin → convert to Celsius
      - Fill: -999
    """
    try:
        with h5py.File(filepath, 'r') as f:

            # ── Read data ─────────────────────────────────────
            if product not in f:
                return None
            data  = f[product][0].copy().astype(np.float32)  # squeeze time
            lat2d = f['Latitude'][:]  * 0.01                 # scale to degrees
            lon2d = f['Longitude'][:] * 0.01

            # ── Attributes ────────────────────────────────────
            attrs    = f[product].attrs
            fill_val = float(attrs.get('_FillValue', [-999.0])[0])
            scale    = float(attrs.get('scale_factor', [1.0])[0])
            offset   = float(attrs.get('add_offset',   [0.0])[0])
            units    = attrs.get('units', b'K')
            units    = units.decode() if isinstance(units, bytes) else str(units)

        # ── Valid India pixels only ────────────────────────────
        # lat2d fill value is 32767*0.01=327.67 — filter those out
        mask = (
            (data != fill_val) &
            (data > 0) &
            (lat2d >= LAT_MIN) & (lat2d <= LAT_MAX) &
            (lon2d >= LON_MIN) & (lon2d <= LON_MAX) &
            (lat2d < 200)  &   # filter fill values in lat
            (lon2d < 200)      # filter fill values in lon
        )

        if mask.sum() == 0:
            return None

        vals = data[mask].astype(np.float64)
        lats = lat2d[mask]
        lons = lon2d[mask]

        # ── Apply scale/offset ────────────────────────────────
        if scale != 1.0 or offset != 0.0:
            vals = vals * scale + offset

        # ── Convert Kelvin → Celsius ──────────────────────────
        # Units are 'K' for LST and SST
        # IMC is rainfall rate (mm/hr) — no conversion needed
        if units == 'K' or (product in ['LST','SST'] and vals.mean() > 100):
            vals = vals - 273.15

        # ── Snap to IMD 0.25° grid ────────────────────────────
        lat_snap = (np.round((lats - LAT_MIN) / 0.25) * 0.25 + LAT_MIN)
        lon_snap = (np.round((lons - LON_MIN) / 0.25) * 0.25 + LON_MIN)
        lat_snap = np.round(lat_snap, 2)
        lon_snap = np.round(lon_snap, 2)

        # Keep only points within valid IMD grid
        valid_lats = set(np.round(LATS_RAIN, 2))
        valid_lons = set(np.round(LONS_RAIN, 2))
        grid_mask  = np.array([
            la in valid_lats and lo in valid_lons
            for la, lo in zip(lat_snap, lon_snap)
        ])

        if grid_mask.sum() == 0:
            return None

        return {
            'lat'  : lat_snap[grid_mask],
            'lon'  : lon_snap[grid_mask],
            'val'  : vals[grid_mask],
            'count': grid_mask.sum()
        }

    except Exception as e:
        return None


def aggregate_insat_day(product: str, date_dir: str) -> pd.DataFrame | None:
    """
    Read all INSAT snapshots for one day → daily mean per 0.25° cell.

    Returns DataFrame with columns:
        latitude, longitude, {product}_mean_c, {product}_std_c
    """
    files = sorted(Path(date_dir).glob('*.h5'))
    if not files:
        return None

    all_lats, all_lons, all_vals = [], [], []

    for f in files:
        result = read_one_insat_file(str(f), product)
        if result is not None:
            all_lats.append(result['lat'])
            all_lons.append(result['lon'])
            all_vals.append(result['val'])

    if not all_lats:
        return None

    # Combine all snapshots
    lats = np.concatenate(all_lats)
    lons = np.concatenate(all_lons)
    vals = np.concatenate(all_vals)

    # Aggregate per grid cell
    df = pd.DataFrame({'lat': lats, 'lon': lons, 'val': vals})
    agg = df.groupby(['lat', 'lon'])['val'].agg(
        mean='mean', std='std', count='count'
    ).reset_index()

    # Rename columns clearly
    suffix = 'c' if product in ['LST','SST'] else 'mmhr'
    agg.rename(columns={
        'lat'  : 'latitude',
        'lon'  : 'longitude',
        'mean' : f'{product}_mean_{suffix}',
        'std'  : f'{product}_std_{suffix}',
        'count': f'{product}_pixel_count',
    }, inplace=True)

    return agg


def read_insat_all_days(year: int, insat_dir: str) -> dict:
    """
    Read all INSAT products for all days → dict of DataFrames.

    Returns:
        {
          'LST': DataFrame(date, latitude, longitude, LST_mean_c, LST_std_c),
          'SST': DataFrame(...),
          'IMC': DataFrame(...),
        }
    """
    n_days = 366 if _is_leap(year) else 365
    dates  = [datetime.date(year, 1, 1) + datetime.timedelta(days=d)
              for d in range(n_days)]

    result = {}

    for product in ['LST', 'SST', 'IMC']:
        print(f"\n  Processing INSAT {product}...")
        product_dir = Path(insat_dir) / product
        frames = []
        missing = 0

        for date in tqdm(dates, desc=f"  {product}"):
            date_str = date.strftime('%Y%m%d')
            date_dir = product_dir / date_str

            if not date_dir.exists():
                missing += 1
                continue

            df = aggregate_insat_day(product, str(date_dir))
            if df is not None:
                df['date'] = str(date)
                frames.append(df)
            else:
                missing += 1

        if frames:
            combined = pd.concat(frames, ignore_index=True)
            result[product] = combined
            print(f"  ✅ {product}: {len(frames)} days processed, "
                  f"{missing} missing, {len(combined):,} rows total")
        else:
            print(f"  ❌ {product}: No data found!")

    return result


# ══════════════════════════════════════════════════════════════
# PART 3 — BUILD MASTER CSV
# ══════════════════════════════════════════════════════════════

def build_master_csv(year: int = 2023):
    """
    Main function — read all data, merge, save master CSV.
    """
    print("=" * 60)
    print(f"Building Master Dataset for year {year}")
    print("=" * 60)

    # ── Step 1: Read IMD ─────────────────────────────────────
    print("\n📂 STEP 1: Reading IMD data...")
    rain, tmax_025, tmin_025 = read_imd_data(year, IMD_DIR)

    # ── Step 2: Build IMD DataFrame ──────────────────────────
    print("\n📊 STEP 2: Building IMD DataFrame...")
    n_days = 366 if _is_leap(year) else 365
    dates  = [datetime.date(year, 1, 1) + datetime.timedelta(days=d)
              for d in range(n_days)]

    imd_rows = []
    for d, date in enumerate(tqdm(dates, desc="  Building IMD rows")):
        date_str = str(date)
        month    = date.month
        doy      = d + 1

        for i, lat in enumerate(LATS_RAIN):
            for j, lon in enumerate(LONS_RAIN):
                r = rain[d, i, j]
                if np.isnan(r):
                    continue   # skip ocean/missing points

                imd_rows.append({
                    'date'        : date_str,
                    'latitude'    : round(float(lat), 2),
                    'longitude'   : round(float(lon), 2),
                    'rainfall_mm' : round(float(r), 4),
                    'temp_max_c'  : round(float(tmax_025[d,i,j]), 4)
                                    if not np.isnan(tmax_025[d,i,j]) else np.nan,
                    'temp_min_c'  : round(float(tmin_025[d,i,j]), 4)
                                    if not np.isnan(tmin_025[d,i,j]) else np.nan,
                    'day_of_year' : doy,
                    'month'       : month,
                    'is_monsoon'  : int(month in [6,7,8,9]),
                })

    df_master = pd.DataFrame(imd_rows)
    print(f"  IMD rows: {len(df_master):,}")
    print(f"  IMD columns: {list(df_master.columns)}")

    # ── Step 3: Read INSAT ───────────────────────────────────
    print("\n🛰️  STEP 3: Reading INSAT data...")
    insat_data = read_insat_all_days(year, INSAT_DIR)

    # ── Step 4: Merge INSAT into master ──────────────────────
    print("\n🔗 STEP 4: Merging INSAT into master...")
    for product, df_insat in insat_data.items():
        before = len(df_master)
        df_master = df_master.merge(
            df_insat,
            on  = ['date', 'latitude', 'longitude'],
            how = 'left'  # keep all IMD points, fill NaN where no INSAT
        )
        after = len(df_master)
        merged = df_master[f'{product}_mean_{"c" if product in ["LST","SST"] else "mmhr"}'].notna().sum()
        print(f"  ✅ {product}: {merged:,}/{len(df_master):,} "
              f"points have INSAT data "
              f"({merged/len(df_master)*100:.1f}%)")

    # ── Step 5: Add next-day targets ─────────────────────────
    print("\n📈 STEP 5: Adding next-day prediction targets...")
    df_master = df_master.sort_values(
        ['latitude', 'longitude', 'date']
    ).reset_index(drop=True)

    # For each location, shift rainfall/temp to get next 1,2,3 days
    df_master['date_dt'] = pd.to_datetime(df_master['date'])

    for days_ahead in [1, 2, 3]:
        df_shifted = df_master[['date_dt','latitude','longitude',
                                 'rainfall_mm','temp_max_c','temp_min_c']].copy()
        df_shifted['date_dt'] = df_shifted['date_dt'] - pd.Timedelta(days=days_ahead)
        df_shifted = df_shifted.rename(columns={
            'rainfall_mm' : f'target_rain_day{days_ahead}',
            'temp_max_c'  : f'target_tmax_day{days_ahead}',
            'temp_min_c'  : f'target_tmin_day{days_ahead}',
        })
        df_master = df_master.merge(
            df_shifted,
            on  = ['date_dt','latitude','longitude'],
            how = 'left'
        )

    df_master.drop(columns=['date_dt'], inplace=True)
    print(f"  ✅ Added prediction targets for days +1, +2, +3")

    # ── Step 6: Final cleanup ─────────────────────────────────
    print("\n🧹 STEP 6: Final cleanup...")

    # Reorder columns logically
    col_order = [
        # Keys
        'date', 'latitude', 'longitude',
        'day_of_year', 'month', 'is_monsoon',
        # IMD inputs (history)
        'rainfall_mm', 'temp_max_c', 'temp_min_c',
        # INSAT inputs
        'LST_mean_c',   'LST_std_c',   'LST_pixel_count',
        'SST_mean_c',   'SST_std_c',   'SST_pixel_count',
        'IMC_mean_mmhr','IMC_std_mmhr', 'IMC_pixel_count',
        # Prediction targets
        'target_rain_day1', 'target_tmax_day1', 'target_tmin_day1',
        'target_rain_day2', 'target_tmax_day2', 'target_tmin_day2',
        'target_rain_day3', 'target_tmax_day3', 'target_tmin_day3',
    ]
    # Only keep columns that exist
    col_order = [c for c in col_order if c in df_master.columns]
    df_master  = df_master[col_order]

    # Sort by date then location
    df_master = df_master.sort_values(
        ['date','latitude','longitude']
    ).reset_index(drop=True)

    # ── Step 7: Validation report ─────────────────────────────
    print("\n📋 STEP 7: Validation Report")
    print("=" * 60)
    print(f"Total rows        : {len(df_master):,}")
    print(f"Total columns     : {len(df_master.columns)}")
    print(f"Date range        : {df_master['date'].min()} "
          f"to {df_master['date'].max()}")
    print(f"Unique dates      : {df_master['date'].nunique()}")
    print(f"Unique grid points: "
          f"{df_master.groupby(['latitude','longitude']).ngroups}")
    print()
    print("Column completeness:")
    for col in df_master.columns:
        pct = df_master[col].notna().mean() * 100
        print(f"  {col:35s}: {pct:6.1f}% complete")
    print()
    print("Value ranges:")
    for col in ['rainfall_mm','temp_max_c','temp_min_c',
                'LST_mean_c','SST_mean_c','IMC_mean_mmhr']:
        if col in df_master.columns:
            mn = df_master[col].min()
            mx = df_master[col].max()
            me = df_master[col].mean()
            print(f"  {col:25s}: {mn:8.2f} to {mx:8.2f} (mean {me:.2f})" )
    print()
    print("Sample rows:")
    print(df_master.head(3).to_string())
    print()
    print("Monsoon vs non-monsoon rainfall:")
    mon = df_master[df_master['is_monsoon']==1]['rainfall_mm']
    non = df_master[df_master['is_monsoon']==0]['rainfall_mm']
    print(f"  Monsoon    mean: {mon.mean():.2f} mm")
    print(f"  Non-monsoon mean:{non.mean():.2f} mm")
    print("  (Monsoon should be higher ✅)"
          if mon.mean() > non.mean() else "  ❌ Check data!")

    # ── Step 8: Save ─────────────────────────────────────────
    print(f"\n💾 STEP 8: Saving to {OUTPUT_CSV}...")
    os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)
    df_master.to_csv(OUTPUT_CSV, index=False)
    size_mb = os.path.getsize(OUTPUT_CSV) / 1e6
    print(f"✅ Saved! File size: {size_mb:.1f} MB")
    print(f"   Rows: {len(df_master):,}")
    print(f"   Columns: {len(df_master.columns)}")
    print()
    print("=" * 60)
    print("✅ MASTER DATASET CREATION COMPLETE!")
    print("=" * 60)

    return df_master


# ══════════════════════════════════════════════════════════════
# UTILITIES
# ══════════════════════════════════════════════════════════════

def _is_leap(year: int) -> bool:
    return (year % 4 == 0 and year % 100 != 0) or (year % 400 == 0)


def verify_single_insat_file(filepath: str, product: str):
    """Quick check of one INSAT file — run before full processing."""
    print(f"\nVerifying: {os.path.basename(filepath)}")
    with h5py.File(filepath, 'r') as f:
        print(f"  Keys     : {list(f.keys())}")
        data  = f[product][0].copy()
        lat2d = f['Latitude'][:]  * 0.01
        lon2d = f['Longitude'][:] * 0.01
        fill  = float(f[product].attrs['_FillValue'][0])
        units = f[product].attrs.get('units', b'?').decode()
        print(f"  Shape    : {data.shape}")
        print(f"  Units    : {units}")
        print(f"  Fill val : {fill}")

        # India pixels
        mask = (
            (data != fill) & (data > 0) &
            (lat2d >= 6.5) & (lat2d <= 38.5) &
            (lon2d >= 66.5) & (lon2d <= 100.0) &
            (lat2d < 200) & (lon2d < 200)
        )
        vals = data[mask]
        if units == 'K':
            vals_c = vals - 273.15
            print(f"  India px : {mask.sum():,}")
            print(f"  Range(K) : {vals.min():.1f} to {vals.max():.1f}")
            print(f"  Range(°C): {vals_c.min():.1f} to {vals_c.max():.1f}")
        else:
            print(f"  India px : {mask.sum():,}")
            print(f"  Range    : {vals.min():.2f} to {vals.max():.2f} {units}")


# ══════════════════════════════════════════════════════════════
# ENTRY POINT
# ══════════════════════════════════════════════════════════════

if __name__ == '__main__':


    parser = argparse.ArgumentParser(description='Create Master Dataset')
    parser.add_argument('--year',    type=int, default=2023)
    parser.add_argument('--verify',  action='store_true',
                        help='Only verify files, do not process')
    # When running in a Colab/Jupyter notebook cell, sys.argv often contains
    # kernel-specific arguments (like -f). Passing an empty list to parse_args
    # tells it to ignore sys.argv and use only declared arguments and their defaults.
    args = parser.parse_args([]) # Fixed for Colab cell execution

    if args.verify:
        # Quick verification of one file per product
        print("Running verification only...")
        for product in ['LST', 'SST', 'IMC']:
            sample_dir = Path(INSAT_DIR) / product
            days = sorted([d for d in sample_dir.iterdir() if d.is_dir()])
            if days:
                files = list(days[0].glob('*.h5'))
                if files:
                    verify_single_insat_file(str(files[0]), product)
    else:
        df = build_master_csv(year=args.year)

Building Master Dataset for year 2023

📂 STEP 1: Reading IMD data...
  Reading rainfall: /content/drive/MyDrive/ISRO/imd/rain/2023.grd
  Rainfall: (365, 129, 135), range 0.0–689.0 mm, valid pts/day: 4964
  Reading tmax: /content/drive/MyDrive/ISRO/imd/tmax/2023.GRD
  Tmax: (365, 31, 31), range 2.7–45.0 °C
  Reading tmin: /content/drive/MyDrive/ISRO/imd/tmin/2023.GRD
  Tmin: (365, 31, 31), range -5.4–31.5 °C
  Regridding temperature to 0.25° grid...


  Regridding: 100%|██████████| 365/365 [00:00<00:00, 476.28it/s]


  Tmax 0.25°: range 3.7–45.0 °C
  Tmin 0.25°: range -5.0–31.5 °C

📊 STEP 2: Building IMD DataFrame...


  Building IMD rows: 100%|██████████| 365/365 [00:28<00:00, 12.63it/s]


  IMD rows: 1,811,781
  IMD columns: ['date', 'latitude', 'longitude', 'rainfall_mm', 'temp_max_c', 'temp_min_c', 'day_of_year', 'month', 'is_monsoon']

🛰️  STEP 3: Reading INSAT data...

  Processing INSAT LST...


...

  ✅ LST: 343 days processed, 22 missing, 2,640,098 rows total

  Processing INSAT SST...


  SST: 100%|██████████| 365/365 [18:29<00:00,  3.04s/it]


  ✅ SST: 343 days processed, 22 missing, 1,347,430 rows total

  Processing INSAT IMC...


  IMC: 100%|██████████| 365/365 [11:43<00:00,  1.93s/it]


  ✅ IMC: 342 days processed, 23 missing, 1,510,824 rows total

🔗 STEP 4: Merging INSAT into master...
  ✅ LST: 1,168,748/1,811,781 points have INSAT data (64.5%)
  ✅ SST: 27,633/1,811,781 points have INSAT data (1.5%)
  ✅ IMC: 355,759/1,811,781 points have INSAT data (19.6%)

📈 STEP 5: Adding next-day prediction targets...
  ✅ Added prediction targets for days +1, +2, +3

🧹 STEP 6: Final cleanup...

📋 STEP 7: Validation Report
Total rows        : 1,811,781
Total columns     : 27
Date range        : 2023-01-01 to 2023-12-31
Unique dates      : 365
Unique grid points: 4964

Column completeness:
  date                               :  100.0% complete
  latitude                           :  100.0% complete
  longitude                          :  100.0% complete
  day_of_year                        :  100.0% complete
  month                              :  100.0% complete
  is_monsoon                         :  100.0% complete
  rainfall_mm                        :  100.0% complete
  temp_m

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv('/content/drive/MyDrive/ISRO/master_dataset_2023.csv')

print("Before filling:")
print(df[['LST_mean_c','SST_mean_c','IMC_mean_mmhr']].isna().mean()*100)

# ── Fix 1: IMC NaN = 0 (no satellite rain = no rain) ─────────
df['IMC_mean_mmhr'] = df['IMC_mean_mmhr'].fillna(0.0)
df['IMC_std_mmhr']  = df['IMC_std_mmhr'].fillna(0.0)

# ── Fix 2: LST NaN = monthly mean for that location ──────────
# Compute monthly mean LST per grid point
monthly_lst = df.groupby(
    ['latitude','longitude','month']
)['LST_mean_c'].transform('mean')
df['LST_mean_c'] = df['LST_mean_c'].fillna(monthly_lst)
df['LST_std_c']  = df['LST_std_c'].fillna(df['LST_std_c'].median())

# ── Fix 3: SST — fill with monthly mean (ocean is slow) ──────
monthly_sst = df.groupby(
    ['latitude','longitude','month']
)['SST_mean_c'].transform('mean')
df['SST_mean_c'] = df['SST_mean_c'].fillna(monthly_sst)
# For coastal/land points with no SST at all — fill with 28°C
# (typical Indian Ocean mean)
df['SST_mean_c'] = df['SST_mean_c'].fillna(28.0)
df['SST_std_c']  = df['SST_std_c'].fillna(0.0)

# ── Fix 4: Temperature NaN (11% missing) ─────────────────────
# Interpolate within each location
df = df.sort_values(['latitude','longitude','date'])
df['temp_max_c'] = df.groupby(
    ['latitude','longitude']
)['temp_max_c'].transform(
    lambda x: x.interpolate(method='linear', limit=5)
)
df['temp_min_c'] = df.groupby(
    ['latitude','longitude']
)['temp_min_c'].transform(
    lambda x: x.interpolate(method='linear', limit=5)
)

# ── Fix 5: Add quality flags ──────────────────────────────────
df['LST_quality']  = (df['LST_pixel_count'] > 50).astype(float).fillna(0)
df['SST_quality']  = (df['SST_pixel_count'] > 10).astype(float).fillna(0)
df['IMC_quality']  = (df['IMC_pixel_count'] > 50).astype(float).fillna(0)

print("\nAfter filling:")
key_cols = ['rainfall_mm','temp_max_c','temp_min_c',
            'LST_mean_c','SST_mean_c','IMC_mean_mmhr']
for col in key_cols:
    pct = df[col].notna().mean() * 100
    print(f"  {col:25s}: {pct:5.1f}% complete")

# Save final version
output = '/content/drive/MyDrive/ISRO/master_dataset_2023_final.csv'
df.to_csv(output, index=False)
print(f"\n✅ Final dataset saved!")
print(f"   Rows   : {len(df):,}")
print(f"   Columns: {len(df.columns)}")
print(f"   Size   : {os.path.getsize(output)/1e6:.1f} MB")

Before filling:
LST_mean_c       35.491762
SST_mean_c       98.474816
IMC_mean_mmhr    80.364128
dtype: float64

After filling:
  rainfall_mm              : 100.0% complete
  temp_max_c               :  89.1% complete
  temp_min_c               :  89.1% complete
  LST_mean_c               :  99.6% complete
  SST_mean_c               : 100.0% complete
  IMC_mean_mmhr            : 100.0% complete

✅ Final dataset saved!
   Rows   : 1,811,781
   Columns: 30
   Size   : 335.1 MB
